#### Name: Blessing Adeniji
#### Degree: MSc Artifical Intelligence Online
#### Capstone Project: AI-Generated Text Detection - Deepfakes

##### Step 1: Data Exploration


In [9]:
# Load all 4 datasets from HuggingFace
# 'load_dataset' is a HuggingFace function that downloads and loads datasets.

from datasets import load_dataset

# Load the 'RAID' dataset
raid_dataset = load_dataset("liamdugan/raid", split="train")

# Load the 'ChatGPT Research Abstracts' dataset
chatgpt_abstracts = load_dataset("NicolaiSivesind/ChatGPT-Research-Abstracts")

# Load the 'GPT-Wiki-Intro' dataset
gpt_wiki_intro_dataset = load_dataset("aadityaubhat/gpt-wiki-intro", split="train")

# Load the 'MAGE' dataset
mage_dataset = load_dataset("yaful/MAGE")

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

In [10]:
# Print the number of rows in each dataset
# len() is a built-in Python function that counts the total number of items
print("RAID:", len(raid_dataset))
print("ChatGPT-Research-Abstracts:", len(chatgpt_abstracts))
print("GPT-Wiki-Intro:", len(gpt_wiki_intro_dataset))
print("MAGE:", len(mage_dataset))

RAID: 5615820
ChatGPT-Research-Abstracts: 1
GPT-Wiki-Intro: 150000
MAGE: 3


In [11]:
# Pandas is a library for working with tabular data in Python.
import pandas as pd

# Convert the datasets to Pandas DataFrames for easier manipulation and analysis
# RAID and GPT-Wiki-Intro dataset directly as single datasets 
raid_df = raid_dataset.to_pandas()
gpt_wiki_intro_df = gpt_wiki_intro_dataset.to_pandas()

abstracts_df = chatgpt_abstracts['train'].to_pandas()
mage_df = mage_dataset['train'].to_pandas() 

# len() counts how many rows are in the table
print("RAID:", len(raid_df))
print("ChatGPT-Research-Abstracts:", len(abstracts_df))
print("GPT-Wiki-Intro:", len(gpt_wiki_intro_df))
print("MAGE:", len(mage_df))

# Print the column names of each dataset
# .coulmns gives column names, .tolist() converts them to a list for easier viewing
print("\nColumns:")
print("RAID:",  raid_df.columns.tolist())
print("ChatGPT-Research-Abstracts:", abstracts_df.columns.tolist())
print("GPT-Wiki-Intro:", gpt_wiki_intro_df.columns.tolist())
print("MAGE:", mage_df.columns.tolist())

RAID: 5615820
ChatGPT-Research-Abstracts: 10000
GPT-Wiki-Intro: 150000
MAGE: 319071

Columns:
RAID: ['id', 'adv_source_id', 'source_id', 'model', 'decoding', 'repetition_penalty', 'attack', 'domain', 'title', 'prompt', 'generation']
ChatGPT-Research-Abstracts: ['title', 'real_abstract', 'real_word_count', 'generated_abstract', 'generated_word_count']
GPT-Wiki-Intro: ['id', 'url', 'title', 'wiki_intro', 'generated_intro', 'title_len', 'wiki_intro_len', 'generated_intro_len', 'prompt', 'generated_text', 'prompt_tokens', 'generated_text_tokens']
MAGE: ['text', 'label', 'src']


In [12]:
# Print the value counts of the 'model' column in the RAID dataset
print(raid_df["model"].value_counts())

model
llama-chat      641808
mpt             641808
mpt-chat        641808
gpt2            641808
mistral         641808
mistral-chat    641808
gpt3            320904
cohere          320904
chatgpt         320904
gpt4            320904
cohere-chat     320904
human           160452
Name: count, dtype: int64


In [13]:
# Create a binary label for the RAID dataset: 1 for AI-generated text, 0 for human-written text
raid_df['label'] = raid_df['model'].apply(lambda x: 0 if x == 'human' else 1)

# Confirms the new label column looks correct
print(raid_df['label'].value_counts())

label
1    5455368
0     160452
Name: count, dtype: int64


In [14]:
# Reshape 'ChatGPT Research Abstracts' into text or label format'
# It currently has 'real_abstract' (human) and 'generated_abstract' (AI) as separate columns.
# Stack them into one column with a label, like RAID and MAGE
# Take human-written abstracts and label them as 0
human_abstracts = abstracts_df[['real_abstract']].copy()
human_abstracts.columns = ['text']
human_abstracts['label'] = 0

# Take AI-generated abstracts and label them as 1
ai_abstracts = abstracts_df[['generated_abstract']].copy()
ai_abstracts.columns = ['text']
ai_abstracts['label'] = 1

# Combine both into a single dataframe like RAID and MAGE
abstracts_reshaped_df = pd.concat([human_abstracts, ai_abstracts], ignore_index=True)

# Count human (0) and AI (1) abstracts in the reshaped dataframe
print(abstracts_reshaped_df['label'].value_counts())

# Print the first few rows
print(abstracts_reshaped_df.head())

label
0    10000
1    10000
Name: count, dtype: int64
                                                text  label
0  This PhD thesis is devoted to deterministic st...      0
1  Phylogenetic approaches are finding more and m...      0
2  Research in Sports Sciences is supported often...      0
3  Let $G$ be a simple, undirected, finite graph ...      0
4  This second part of a 2 volume-expertise is ma...      0


In [15]:
# Reshape GPT-Wiki-Intro dataset into text or label format
# 'gpt_wiki_intro_df' = human-written, 'generated_intro' = AI-generated
#  Human wikipedia introductions are labeled as 0
human_wiki = gpt_wiki_intro_df[['wiki_intro']].copy()
human_wiki.columns = ['text']
human_wiki['label'] = 0

# AI-generated introductions are labeled as 1
ai_wiki = gpt_wiki_intro_df[['generated_intro']].copy()
ai_wiki.columns = ['text']
ai_wiki['label'] = 1

# Combine both into a single dataframe like RAID and MAGE
wiki_intro_reshaped_df = pd.concat([human_wiki, ai_wiki], ignore_index=True)

# Count human (0) and AI (1) introductions in the reshaped dataframe
print(wiki_intro_reshaped_df['label'].value_counts())

# Print the first few rows
print(wiki_intro_reshaped_df.head())


label
0    150000
1    150000
Name: count, dtype: int64
                                                text  label
0  Sexhow railway station was a railway station b...      0
1  In Finnish folklore, all places and things, an...      0
2  In mathematics, specifically differential calc...      0
3  is a Japanese shōjo manga series written and i...      0
4  Robert Milner "Rob" Bradley, Jr. (born August ...      0


In [16]:
# Text length analysis across all datasets
# Prepare RAID dataset in text or label format
raid_resharped_df = raid_df[['generation', 'label']].copy() 
raid_resharped_df.columns = ['text', 'label']

# loop all dataset in one dictionary for easier processing
datasets = {
    "RAID": raid_resharped_df,
    "ChatGPT-Research-Abstracts": abstracts_reshaped_df,
    "GPT-Wiki-Intro": wiki_intro_reshaped_df,
    "MAGE": mage_df,
}


    
# for each dataset, calculate the length of the text and print the average length
for name, df in datasets.items():
    # count words in each text entry
    lengths = df['text'].astype(str).apply(lambda x: len(x.split()))

    # print the name of dataset
    print(f"{name}")

    # print the average word count
    print("Average length:", round(lengths.mean(), 1), "words")

    # print the median word count
    print("Median length:", lengths.median(), "words")

    # print the maximum word count
    print("Maximum length:", lengths.max(), "words")

    # How many texts are longer than 512 model input limit
    print("Texts longer than 512 words:", (lengths > 512).sum())



RAID
Average length: 235.5 words
Median length: 231.0 words
Maximum length: 13261 words
Texts longer than 512 words: 14766
ChatGPT-Research-Abstracts
Average length: 191.5 words
Median length: 187.0 words
Maximum length: 584 words
Texts longer than 512 words: 11
GPT-Wiki-Intro
Average length: 163.3 words
Median length: 165.0 words
Maximum length: 447 words
Texts longer than 512 words: 0
MAGE
Average length: 211.5 words
Median length: 115.0 words
Maximum length: 10090 words
Texts longer than 512 words: 38391


In [17]:
# Class imbalance analysis and duplicate detection across all datasets
for name, df in datasets.items():
    # print the name of dataset
    print(f"{name}")

    # count the number of human (0) and AI (1) texts
    print("Class distribution:")
    print(df['label'].value_counts())

    # count how many texts are duplicates
    duplicates = df['text'].duplicated().sum()
    print("Duplicate texts:", duplicates)

RAID
Class distribution:
label
1    5455368
0     160452
Name: count, dtype: int64
Duplicate texts: 640246
ChatGPT-Research-Abstracts
Class distribution:
label
0    10000
1    10000
Name: count, dtype: int64
Duplicate texts: 0
GPT-Wiki-Intro
Class distribution:
label
0    150000
1    150000
Name: count, dtype: int64
Duplicate texts: 0
MAGE
Class distribution:
label
0    225753
1     93318
Name: count, dtype: int64
Duplicate texts: 0


In [18]:
# Clean RAID dataset by removing duplicates and texts longer than 512 words, then balance the classes
# Remove duplicates, keep first occurrence
raid_cleaned_df = raid_resharped_df.drop_duplicates(subset='text').copy()
print("RAID after removing duplicates:", len(raid_cleaned_df))

# Split into human and AI texts
raid_human_df = raid_cleaned_df[raid_cleaned_df['label'] == 0]
raid_ai_df = raid_cleaned_df[raid_cleaned_df['label'] == 1]

# Randomly sample AI texts to match the size of the human class
# random_state=42 ensures reproducibility of the random sampling (same result every run)
raid_ai_sampled_df = raid_ai_df.sample(n=len(raid_human_df), random_state=42)

# Combine into a balanced dataset and shuffle the rows
raid_balanced_df = pd.concat([raid_human_df, raid_ai_sampled_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# print the class distribution of the balanced dataset
print("RAID balanced class distribution:")
print(raid_balanced_df['label'].value_counts())


RAID after removing duplicates: 4975574
RAID balanced class distribution:
label
1    146735
0    146735
Name: count, dtype: int64


In [19]:
# Balance MAGE dataset which has 225k human but only 93k AI texts - unfair mix.
# Solution: Randomly keep only 93k human texts to match the AI class size, then shuffle the rows
# Separate the human and AI texts
mage_human_df = mage_df[mage_df['label'] == 0]
mage_ai_df = mage_df[mage_df['label'] == 1]

# Randomly sample human texts to match the size of the AI class
# random_state=42 ensures reproducibility of the random sampling (same result every run)
mage_human_sampled_df = mage_human_df.sample(n=len(mage_ai_df), random_state=42)

# Combine into a balanced dataset and shuffle the rows
mage_balanced_df = pd.concat([mage_human_sampled_df, mage_ai_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# print the class distribution of the balanced dataset
print("MAGE balanced class distribution:")
print(mage_balanced_df['label'].value_counts())

MAGE balanced class distribution:
label
1    93318
0    93318
Name: count, dtype: int64


In [20]:
# 70/15/15 train/validation/test split for all datasets
# train = model learning, validation = hyperparameter tuning, test = final evaluation

from sklearn.model_selection import train_test_split
import os

# Folder to save the split datasets
os.makedirs("data_splits", exist_ok=True)

# Final cleaned versions of all 4 datasets
final_datasets = {
    "RAID": raid_balanced_df,
    "ChatGPT-Research-Abstracts": abstracts_reshaped_df,
    "GPT-Wiki-Intro": wiki_intro_reshaped_df,
    "MAGE": mage_balanced_df,
}

for name, df in final_datasets.items():
    # Split into train (70%) and 30% ramaining
    # stratify=df['label'] ensures that the class distribution is preserved in the splits
    train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)

    # second split: Split the remaining 30% into validation (15%) and test (15%)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

    # save each split as a CSV file in the "data_splits" folder
    train_df.to_csv(f"data_splits/{name}_train.csv", index=False)
    val_df.to_csv(f"data_splits/{name}_val.csv", index=False)
    test_df.to_csv(f"data_splits/{name}_test.csv", index=False)

    # print the sizes of each split
    print(f"{name} - Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")

RAID - Train: 205429, Validation: 44020, Test: 44021
ChatGPT-Research-Abstracts - Train: 14000, Validation: 3000, Test: 3000
GPT-Wiki-Intro - Train: 210000, Validation: 45000, Test: 45000
MAGE - Train: 130645, Validation: 27995, Test: 27996
